In [ ]:
from js import location
import micropip

_path = location.pathname
_root = _path.split("/extensions/")[0] if "/extensions/" in _path else _path.rsplit("/", 1)[0]
_pypi = location.origin + _root + "/pypi/"
_wheels = [
    'lark-1.3.1-py3-none-any.whl',
    'icontract-2.7.3-py3-none-any.whl',
    'asttokens-2.4.1-py2.py3-none-any.whl',
    'typing_extensions-4.16.0-py3-none-any.whl',
    'six-1.17.0-py2.py3-none-any.whl',
    'comm-0.2.3-py3-none-any.whl',
    'fpl_walker-0.0.0-py3-none-any.whl',
]
await micropip.install([_pypi + w for w in _wheels], deps=False)
import fpl
fpl


In [ ]:
import marimo as mo

# 01.04.01 Pass

Python has no pipe operator. `>>` on a Pass is composition. On an int it stays a shift.

A pass is an arrow between two representations, named on the pass pages
(`Source -lex-> Tokens`). `source` and `target` carry those names as plain
strings: `"Source"`, `"Tokens"`, `"Syntax"`, `"Core"`, `"Stacks"`. They are the
keys a codec is looked up by when a pass runs across the pipe. An empty name
means unknown, and an unknown end matches anything.

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, cast


@dataclass(frozen=True)
class Pass[A, B]:
    """
    One arrow from `source` to `target`. Calling it runs `fn`; `>>`
    composes it with the next pass into a PassCompose.
    """

    name: str
    fn: Callable[[A], B]
    source: str = ""
    target: str = ""

    def __call__(self, value: A) -> B:
        return self.fn(value)

    def __rshift__[C](
        self, nxt: Pass[B, C] | PassCompose[B, C]
    ) -> PassCompose[A, C]:
        return cast(PassCompose[A, C], pass_compose(self, nxt))

## Composition is data

`a >> b` used to return a new Pass whose function called the two in turn, so
the chain survived only as a name
([TD-0008](../../02-docs/02.05-project/techdebt/TD-0008-pass-composition-opaque-closure.md)).
A `PassCompose` keeps the passes themselves, in order, as a flat tuple: it is
the "then" of two computations, normalised so that nesting never shows.
`(a >> b) >> c` and `a >> (b >> c)` are the same value. The stages can be
listed, printed and checked before anything runs, and any executor that can
run a pass can run the whole: `direct` below calls them one after the other in
this process; the shell pipe runs each in its own process
([ADR-0012](../../02-docs/02.05-project/adr/ADR-0012-pipeline-is-a-computation-graph.md)).

`>>` refuses to join two passes whose representations are both known and
differ, so `parse >> lex` is an error where it is written, not where it runs.

In [ ]:
type Stage = Pass[Any, Any]


@dataclass(frozen=True)
class PassCompose[A, B]:
    """
    Passes run in order, the output of each the input of the next.

    `stages` is flat and never empty, and each stage writes what the next
    reads; the constructor refuses anything else. Build it with
    `pass_compose` or `>>`, which also flatten nested compositions.
    Calling the value runs it with `direct`.
    """

    stages: tuple[Stage, ...]

    def __post_init__(self) -> None:
        if not self.stages:
            raise ValueError("a PassCompose needs at least one pass")
        for left, right in zip(self.stages, self.stages[1:]):
            if left.target and right.source and left.target != right.source:
                raise TypeError(
                    f"{left.name} writes {left.target}, "
                    f"{right.name} reads {right.source}"
                )

    @property
    def name(self) -> str:
        """`lex>>parse`: the stage names joined by `>>`."""
        return ">>".join(stage.name for stage in self.stages)

    @property
    def source(self) -> str:
        """The representation the first stage reads."""
        return self.stages[0].source

    @property
    def target(self) -> str:
        """The representation the last stage writes."""
        return self.stages[-1].target

    def __call__(self, value: A) -> B:
        return cast(B, direct(self, value))

    def __rshift__[C](
        self, nxt: Pass[B, C] | PassCompose[B, C]
    ) -> PassCompose[A, C]:
        return cast(PassCompose[A, C], pass_compose(self, nxt))

    def __repr__(self) -> str:
        names = ", ".join(stage.name for stage in self.stages)
        return f"pass_compose({names})"


def pass_compose(
    *parts: Stage | PassCompose[Any, Any],
) -> PassCompose[Any, Any]:
    """
    The composition of `parts`, in order, with every nested PassCompose
    spliced in.

    Raises ValueError for no parts, and TypeError where one stage writes
    a representation the next does not read (both named, and different).
    """
    stages: list[Stage] = []
    for part in parts:
        if isinstance(part, PassCompose):
            stages.extend(part.stages)
        else:
            stages.append(part)
    return PassCompose(tuple(stages))


def direct(computation: Stage | PassCompose[Any, Any], value: Any) -> Any:
    """
    Run `computation` on `value` in this process: each stage in turn,
    with no copy and no serialisation between them.
    """
    stages = (
        computation.stages
        if isinstance(computation, PassCompose)
        else (computation,)
    )
    for stage in stages:
        value = stage(value)
    return value